# Kaggle environment diagnostic

This notebook is intentionally small. It does not install packages, download models, or use secrets.

In [ ]:
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from pathlib import Path


def run_command(command):
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=20)
        return {
            'returncode': result.returncode,
            'stdout': result.stdout.strip(),
            'stderr': result.stderr.strip(),
        }
    except Exception as exc:
        return {'error': f'{type(exc).__name__}: {exc}'}


def read_meminfo():
    values = {}
    try:
        for line in Path('/proc/meminfo').read_text().splitlines():
            match = re.match(r'^(MemTotal|MemAvailable):\s+(\d+) kB$', line)
            if match:
                values[match.group(1)] = int(match.group(2)) * 1024
    except Exception as exc:
        values['error'] = f'{type(exc).__name__}: {exc}'
    return values


report = {
    'python': sys.version,
    'executable': sys.executable,
    'platform': platform.platform(),
    'machine': platform.machine(),
    'processor': platform.processor(),
    'cpu_count': os.cpu_count(),
    'kaggle_working_dir_exists': Path('/kaggle/working').exists(),
    'meminfo_bytes': read_meminfo(),
    'disk_bytes': {},
    'nvidia_smi': run_command(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader']),
    'torch': {},
    'internet_kaggle': {},
}

disk = shutil.disk_usage('/')
report['disk_bytes'] = {'total': disk.total, 'used': disk.used, 'free': disk.free}

if importlib.util.find_spec('torch') is not None:
    try:
        import torch
        report['torch'] = {
            'version': torch.__version__,
            'cuda_available': bool(torch.cuda.is_available()),
            'cuda_version': torch.version.cuda,
            'cudnn_version': torch.backends.cudnn.version(),
            'device_count': torch.cuda.device_count(),
        }
        if torch.cuda.is_available():
            report['torch']['devices'] = []
            for index in range(torch.cuda.device_count()):
                report['torch']['devices'].append({
                    'index': index,
                    'name': torch.cuda.get_device_name(index),
                    'total_memory_bytes': torch.cuda.get_device_properties(index).total_memory,
                })
            test_tensor = torch.ones((32, 32), device='cuda')
            expected = torch.full((32, 32), 32.0, device='cuda')
            report['torch']['small_cuda_test'] = {
                'passed': bool(torch.equal(test_tensor @ test_tensor, expected)),
                'allocated_bytes': torch.cuda.memory_allocated(),
            }
    except Exception as exc:
        report['torch']['error'] = f'{type(exc).__name__}: {exc}'
else:
    report['torch']['status'] = 'torch is not installed in the base environment'

try:
    from urllib.request import urlopen
    with urlopen('https://www.kaggle.com', timeout=15) as response:
        report['internet_kaggle'] = {'status_code': response.status}
except Exception as exc:
    report['internet_kaggle'] = {'error': f'{type(exc).__name__}: {exc}'}

output_path = Path('/kaggle/working/kaggle_environment_report.json')
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(report, indent=2, default=str) + '\n')
print(json.dumps(report, indent=2, default=str))
print(f'\nSaved report to {output_path}')